In [1]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import col

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MMDS") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/27 17:27:06 WARN Utils: Your hostname, MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.234 instead (on interface en0)
25/12/27 17:27:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/27 17:27:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Define schema and dataframes

In [3]:
schema_ratings = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("item_id", IntegerType(), False),
    StructField("rating", IntegerType(), False),
    StructField("timestamp", IntegerType(), False)
])

schema_movies = StructType([
    StructField("item_id", IntegerType(), False),
    StructField("title", StringType(), False),
    StructField('genres', StringType(), False)
])

schema_users = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("gender", StringType(), False),
    StructField('age', StringType(), False),
    StructField('occupation', IntegerType(), False),
    StructField('zip_code', StringType(), False)
])

In [4]:
train_ratings = spark.read.option("delimiter", "::").csv("./data/ratings_train.dat", schema=schema_ratings)
test_ratings = spark.read.option("delimiter", "::").csv("./data/ratings_test.dat", schema=schema_ratings)

In [5]:
movies = spark.read.option("delimiter", "::").csv("./data/movies.dat", schema=schema_movies)
movies = (
    movies
    .withColumn("year", substring("title", -5, 4).cast("int"))
    .withColumn("title", substring("title", 0, length("title") - 6))
    .withColumn("genres", split("genres", r"\|"))
)
movies.printSchema()

root
 |-- item_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- year: integer (nullable = true)



In [6]:
users = spark.read.option("delimiter", "::").csv("./data/users.dat", schema=schema_users)

users = (
    users
    .withColumn("gender", when(col("gender") == "F", 0).otherwise(1))
)

users.show()

+-------+------+---+----------+--------+
|user_id|gender|age|occupation|zip_code|
+-------+------+---+----------+--------+
|      1|     0|  1|        10|   48067|
|      2|     1| 56|        16|   70072|
|      3|     1| 25|        15|   55117|
|      4|     1| 45|         7|   02460|
|      5|     1| 25|        20|   55455|
|      6|     0| 50|         9|   55117|
|      7|     1| 35|         1|   06810|
|      8|     1| 25|        12|   11413|
|      9|     1| 25|        17|   61614|
|     10|     0| 35|         1|   95370|
|     11|     0| 25|         1|   04093|
|     12|     1| 25|        12|   32793|
|     13|     1| 45|         1|   93304|
|     14|     1| 35|         0|   60126|
|     15|     1| 25|         7|   22903|
|     16|     0| 35|         0|   20670|
|     17|     1| 50|         1|   95350|
|     18|     0| 18|         3|   95825|
|     19|     1|  1|        10|   48073|
|     20|     1| 25|        14|   55113|
+-------+------+---+----------+--------+
only showing top

## Movies profile

In [7]:
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.ml.feature import Normalizer

cv = CountVectorizer(
    inputCol="genres",
    outputCol="tf",
    vocabSize=18,
    minDF=1
)

cv_model = cv.fit(movies)
tf_df = cv_model.transform(movies)

idf = IDF(inputCol="tf", outputCol="tfidf")
idf_model = idf.fit(tf_df)

movie_tfidf = idf_model.transform(tf_df)

normalizer = Normalizer(
    inputCol="tfidf",
    outputCol="features_norm",
    p=2
)

movies_profiles = normalizer.transform(movie_tfidf)

## Users profile

In [14]:
from pyspark.ml.stat import Summarizer
from pyspark.ml.feature import Normalizer

movie_vecs = movies_profiles.select(
    col("item_id"),
    col("features_norm")
)

user_movie_vectors = (
    train_ratings
    .filter(col("rating") >= 4)
    .join(broadcast(movie_vecs), on='item_id') ## broadcase here so that movie_vecs is moved to every executor, instead of shuffling ratings
    .select("user_id", "rating", "features_norm")
)

user_profiles = (
    user_movie_vectors
    .groupBy("user_id")
    .agg(
        Summarizer.mean(
            col("features_norm"),
        ).alias("user_features")
    )
)

normalizer = Normalizer(
    inputCol="user_features",
    outputCol="user_features_norm",
    p=2
)

user_profiles = normalizer.transform(user_profiles)

## LSH

In [15]:
from pyspark.ml.feature import BucketedRandomProjectionLSH

lsh = BucketedRandomProjectionLSH(
    inputCol="features_norm",
    outputCol="hashes",
    bucketLength=0.1,
)

lsh_model = lsh.fit(movies_profiles)

movies_lsh = movies_profiles.select(
    col("item_id"),
    col("features_norm")
).cache()

users_lsh = user_profiles.select(
    col("user_id"),
    col("user_features_norm").alias("features_norm")
).cache()

25/12/27 17:30:35 WARN CacheManager: Asked to cache already cached data.


In [16]:
from pyspark import StorageLevel

recommendations = lsh_model.approxSimilarityJoin(
    users_lsh, 
    movies_lsh, 
    threshold=1,
    distCol="distance"
).select(
    col("datasetA.user_id").alias("user_id"),
    col("datasetB.item_id").alias("item_id"),
    col("distance")
)

In [17]:
already_rated = train_ratings.select("user_id", "item_id")
already_rated.count()

802553

In [18]:
from pyspark.sql.window import Window

recommendations = recommendations.join(
    already_rated,
    on=["user_id", "item_id"],
    how="left_anti"
)

window = Window.partitionBy("user_id").orderBy(col("distance").asc())
top_k = 1000

ranked_recs = (
    recommendations
    .withColumn("rank", row_number().over(window))
    .filter(col("rank") <= top_k)
)

In [19]:
rated_universe = (
    test_ratings
    .select("user_id", "item_id", "rating")
)

recs_on_rated = (
    ranked_recs
    .join(rated_universe, on=["user_id", "item_id"], how="inner")
)

eval_df = (
    recs_on_rated
    .withColumn("relevant", (col("rating") >= 4).cast("int"))
)

user_metrics = (
    eval_df
    .groupBy("user_id")
    .agg(
        (sum("relevant") / lit(top_k)).alias("precision"),
        sum("relevant").alias("hits"),
        count("*").alias("rated_recommended")
    )
    .join(
        rated_universe
        .filter(col("rating") >= 4)
        .groupBy("user_id")
        .count()
        .withColumnRenamed("count", "total_relevant"),
        on="user_id",
        how="left"
    )
    .fillna(0)
    .withColumn(
        "recall",
        when(col("total_relevant") > 0,
             col("hits") / col("total_relevant"))
        .otherwise(lit(0))
    )
)

avg_metrics = user_metrics.agg(
    avg("precision").alias(f"avg_precision@{top_k}"),
    avg("recall").alias(f"avg_recall@{top_k}")
)

avg_metrics.show()

+--------------------+-------------------+
|  avg_precision@1000|    avg_recall@1000|
+--------------------+-------------------+
|0.002315019937970...|0.17301327303075115|
+--------------------+-------------------+

